In [5]:
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
DataCollatorForLanguageModeling, TrainingArguments, Trainer)

from peft import LoraConfig, get_peft_model
# Select model
model_name_path = "models/gemma_3.1_4B_instruct"

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16
# Load tokenizer
tok = AutoTokenizer.from_pretrained(model_name_path, use_fast=True)
tok.pad_token = tok.eos_token


In [6]:
# Load model
model = AutoModelForCausalLM.from_pretrained(model_name_path, dtype=torch.float16)
model.to(device)

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.79s/it]


Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [7]:
args = TrainingArguments(
output_dir="output",
per_device_train_batch_size=1,
gradient_accumulation_steps=8,
learning_rate=2e-4,
num_train_epochs=1,
fp16=False,
bf16=False,
dataloader_pin_memory=False,
logging_steps=10,
save_total_limit=2,
report_to="none"
)

In [9]:
# Memory optimization for MPS
model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

In [10]:
# LoRA config
lora_cfg = LoraConfig(
r=8, lora_alpha=16, lora_dropout=0.05,
target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_cfg)

In [11]:
model = get_peft_model(model, lora_cfg)
# Dataset (place your own my_data.json in format [{"instruction": "", "input": "", "output": ""}, {....}])


/Users/gjorgjinoveski/PycharmProjects/llm_fine_tune/.venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/gjorgjinoveski/PycharmProjects/llm_fine_tune/.venv/lib/python3.12/site-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'models/gemma_3.1_4B_instruct' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/Users/gjorgjinoveski/PycharmProjects/llm_fine_tune/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
ds = load_dataset("json", data_files="my_data.json", split="train")
def format_example(data):
prompt = f"Instruction: {data['instruction']}\nInput: {data['input']}\nResponse:"
text = prompt + data["output"] + tok.eos_token
return {"input_ids": tok(text, truncation=True, max_length=1024)["input_ids"]}
ds = ds.map(format_example, remove_columns=["instruction", "input", "output"])
collator = DataCollatorForLanguageModeling(tok, mlm=False)
trainer = Trainer(
model=model,
args=args,
train_dataset=ds,
data_collator=collator,
)
trainer.train()
# Save LoRA adapter
model.save_pretrained("out/lora")
tok.save_pretrained("out/lora")
print("Done. LoRA adapter saved to out/lora")